# Task 3 — Gender MixUp screen

Run All trains **folds 0 and 4 only**, from scratch. Start from completed **04ad**, with its name-based labels, dropout **0.30**, grayscale probability **0.10**, and mild darkening.
The only new factor is **MixUp with alpha 0.2**: blend two training images and their labels. Use plain, unweighted cross-entropy.

Use a **fresh Colab L4** matching the previous runs. Push this notebook and its source files before running. Completed 04ad runs, their earlier parents and 04w precision evidence must be on Drive.


## 1. Colab GPU and repository


In [ ]:
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

REPO_URL = "https://github.com/TrnLin/MLA2.git"
BRANCH = "task-3-gender-usage-classification"
REPO_DIR = Path("/content/MLA2")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT_DIR = DRIVE_MOUNT / "MyDrive/MLA2"
DATA_ZIP = DRIVE_PROJECT_DIR / "data/task3-data.zip"
LOCAL_DATA_ZIP = Path("/content/task3-data.zip")
DRIVE_TASK_DIR = DRIVE_PROJECT_DIR / "task3"
DRIVE_REGISTRY = DRIVE_TASK_DIR / "results/runs.csv"
LOCAL_REGISTRY = REPO_DIR / "results/runs.csv"


def run_checked(command, *, cwd=None):
    command = [str(part) for part in command]
    print("$", " ".join(command), flush=True)
    return subprocess.run(command, cwd=cwd, check=True)


try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("Connect this notebook to a Google Colab GPU runtime first.") from exc

drive.mount(str(DRIVE_MOUNT), force_remount=False)
if (REPO_DIR / ".git").is_dir():
    remote_url = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True
    ).strip()
    if remote_url != REPO_URL:
        raise RuntimeError(f"{REPO_DIR} belongs to a different repository: {remote_url}")
    run_checked(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    run_checked(["git", "switch", BRANCH], cwd=REPO_DIR)
    run_checked(["git", "merge", "--ff-only", f"origin/{BRANCH}"], cwd=REPO_DIR)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository.")
else:
    run_checked(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR])

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print("Repository ready:", REPO_DIR)
print("Commit:", commit)

## 2. Teacher data and canonical split


In [ ]:
def copy_teacher_zip_to_local_disk():
    if LOCAL_DATA_ZIP.is_file():
        try:
            with zipfile.ZipFile(LOCAL_DATA_ZIP) as existing:
                existing.infolist()
            return
        except zipfile.BadZipFile:
            LOCAL_DATA_ZIP.unlink()
    partial = LOCAL_DATA_ZIP.with_suffix(".zip.partial")
    for attempt in range(1, 4):
        partial.unlink(missing_ok=True)
        try:
            expected_bytes = DATA_ZIP.stat().st_size
            with DATA_ZIP.open("rb") as source, partial.open("wb") as target:
                shutil.copyfileobj(source, target, length=8 * 1024**2)
            if partial.stat().st_size != expected_bytes:
                raise OSError("The local ZIP copy is incomplete.")
            partial.replace(LOCAL_DATA_ZIP)
            return
        except OSError as error:
            partial.unlink(missing_ok=True)
            if attempt == 3:
                raise RuntimeError("Drive disconnected three times. Remount and retry.") from error
            drive.mount(str(DRIVE_MOUNT), force_remount=True)
            time.sleep(2)


copy_teacher_zip_to_local_disk()
teacher_dir = REPO_DIR / "data/raw/teacher"
required_files = (
    teacher_dir / "train/styles_train.csv",
    teacher_dir / "test/styles_prediction.csv",
)
image_dirs = (teacher_dir / "train/images_train", teacher_dir / "test/images_test")
image_suffixes = {".jpg", ".jpeg"}
with zipfile.ZipFile(LOCAL_DATA_ZIP) as archive:
    names = archive.namelist()
    if any(Path(name).is_absolute() or ".." in Path(name).parts for name in names):
        raise RuntimeError("The teacher archive contains an unsafe path.")
    expected_images = sum(
        name.startswith("data/raw/teacher/") and Path(name).suffix.lower() in image_suffixes
        for name in names
    )
    current_images = sum(
        path.suffix.lower() in image_suffixes for folder in image_dirs for path in folder.glob("*")
    )
    if current_images != expected_images or not all(path.is_file() for path in required_files):
        archive.extractall(REPO_DIR)

actual_images = sum(
    path.suffix.lower() in image_suffixes for folder in image_dirs for path in folder.glob("*")
)
if actual_images != expected_images or not all(path.is_file() for path in required_files):
    raise RuntimeError(f"Teacher data is incomplete: {actual_images:,}/{expected_images:,} images")

os.chdir(REPO_DIR)
os.environ["FASHION_PROJECT_ROOT"] = str(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
(DRIVE_TASK_DIR / "results").mkdir(parents=True, exist_ok=True)
print(f"Teacher data ready: {actual_images:,} images")

## 3. Freeze MixUp and check the parents

The weighting screen reduced the mean clean gap by only 0.024 versus G2; the required reduction was 0.050. MixUp tests whether learning from blended examples reduces memorization. Start from unweighted **04ad**, so this trial changes one factor.

For **every training batch**, draw one value `lambda ~ Beta(0.2, 0.2)` and randomly shuffle the rows to choose partners. Self-pairs and same-label pairs are allowed. After the existing image augmentation and RGB normalization, use:

- Image: `lambda * image + (1 - lambda) * partner_image`.
- Loss: `lambda * CE(logits, label) + (1 - lambda) * CE(logits, partner_label)`.

Alpha 0.2 usually puts more weight on one image. The separate PCG64 random stream uses `2753 XOR 0x4D495855`, persists across epochs, and does not draw from the augmentation or dropout random streams. Partners come only from the current training batch. Every original training row appears once per epoch. Keep labels and the canonical split fixed.

Keep widths `[32, 64, 128, 256]`, **390,181 parameters**, full images, GeM p=3, dropout 0.30, clean fold-training RGB normalization, AdamW weight decay 0.0001, the G2 learning rate and cosine schedule, batch 128, **30 epochs**, seed **2753**, and the final-epoch checkpoint.

Augmentation stays: translation ±2 px with probability 0.50, mild darkening with probability 0.25 and brightness factor 0.90–1.00, then RGB → L → RGB with probability 0.10. Models start with random weights; saved parents are comparison evidence.

Blended images have blended labels, so ordinary online training F1 is not meaningful. Save mixed training loss and leave online training F1 blank. **The gap uses clean, unmixed training and validation images from the final model in evaluation mode**, with plain unweighted loss and no dropout or random training augmentation. All corruption evaluations are unmixed too.

G2, E6, Gray10, completed 04ad and new models are compared on identical name-truth labels using IEEE FP32. Keep the original teacher-label diagnostic. MixUp can hide small product cues, so read class scores and corruption results alongside the gap.

This is an explicit MixUp exception for this separate trial to the earlier frozen no-MixUp plan. It does not alter the earlier experiments. It was chosen after looking at development results; it is not an independent blind test or an accepted submission model.


In [ ]:
from fashion.data.gender_name_truth import build_gender_name_truth_variant
from fashion.train.task3_gender_mixup import (
    check_gender_mixup_sources,
    run_gender_mixup_screen,
)

G2_DIR = DRIVE_TASK_DIR / "experiments/t3_gender_v2_g2_translation/gender"
E6_DIR = DRIVE_TASK_DIR / "experiments/t3_gender_e6_gem_p3/gender"
DROPOUT_DIR = DRIVE_TASK_DIR / "experiments/t3_gender_dropout_030/gender"
DARKENING_DIR = DRIVE_TASK_DIR / "experiments/t3_gender_dropout_030_mild_darkening/gender"
GRAYSCALE_DIR = (
    DRIVE_TASK_DIR / "experiments/t3_gender_dropout_030_mild_darkening_grayscale_010/gender"
)
NAME_TRUTH_DIR = (
    DRIVE_TASK_DIR / "experiments/t3_gender_name_truth_dropout_030_grayscale_010/gender"
)
PRECISION_DIR = DRIVE_TASK_DIR / "diagnostics/gender_precision/20260905T085822668071Z"

summary = build_gender_name_truth_variant(REPO_DIR)
print("Changed gender labels:", summary["changed_labels"])
print("Unclear names kept:", summary["no_cue_rows"] + summary["multiple_cue_rows"])
print("Fold label changes:", summary["folds"])
sources, classes, spec, evidence = check_gender_mixup_sources(
    g2_directory=G2_DIR,
    e6_directory=E6_DIR,
    dropout_directory=DROPOUT_DIR,
    darkening_directory=DARKENING_DIR,
    grayscale_directory=GRAYSCALE_DIR,
    name_truth_directory=NAME_TRUTH_DIR,
    source_registry_path=DRIVE_REGISTRY,
    precision_directory=PRECISION_DIR,
    root=REPO_DIR,
)
assert spec.classifier_dropout == 0.30
assert spec.to_dict()["grayscale_probability"] == 0.10
print("Verified source runs:", {name: len(runs) for name, runs in sources.items()})
print(
    "Direct name-truth parents:",
    {fold: run["run_id"] for fold, run in sources["NameTruth"].items()},
)
print("Frozen recipe:", spec.to_dict())
print("Output:", DRIVE_TASK_DIR / spec.artifact_dir / "gender")
print("MixUp policy:", spec.to_dict()["mixup_policy"])


## 4. Unchanged screen rules

All rules must pass, using the same full-FP32 evaluation and the same name-truth labels for each model:

- Pooled validation macro-F1 falls by at most **0.030 versus matched G2**. The paired whole-family bootstrap 95% lower bound for the difference must be **at least −0.030**. Neither fold may lose more than 0.030. Use 10,000 draws within folds, seed 2753.
- The mean clean training–validation F1 gap falls by at least **0.050**. **Both folds' gaps must shrink.** Use clean evaluation-mode training scores from each finished checkpoint, not online augmented training scores.
- Preserve the existing stricter class guard: no pooled class loses more than **0.020 F1**. Thus the overall 0.030 allowance does not override a class failure. NLL may rise by at most 0.020 and ECE by at most 0.010 versus G2; these measure the quality of model confidence.
- Preserve corruption guards versus matched E6: the translation-induced F1 change improves by at least 0.030; every other standard corruption, including darkening, worsens by at most 0.020. Each corrupted score is measured relative to that model's clean score.
- Exactly **390,181 parameters** and peak allocated GPU memory **strictly below 3,000,000,000 bytes**. Training time and latency are reported without speed caps. A memory failure stops before another fold begins.

The thresholds are derived from the new matched IEEE reference scores; do not substitute the rounded historical values. A pass means this screen met the chosen trade-off, not that the model is accepted or more accurate.


Also report the direct change versus the completed **04ad NameTruth** parent: pooled and per-fold F1, class F1, clean gaps, confidence quality and raw corruption scores. This shows what MixUp adds with labels held fixed. It adds no new acceptance gate. Do not call a smaller training gap a win if validation or rare-class scores get worse.

In [ ]:
result = run_gender_mixup_screen(
    g2_directory=G2_DIR,
    e6_directory=E6_DIR,
    dropout_directory=DROPOUT_DIR,
    darkening_directory=DARKENING_DIR,
    grayscale_directory=GRAYSCALE_DIR,
    name_truth_directory=NAME_TRUTH_DIR,
    source_registry_path=DRIVE_REGISTRY,
    precision_directory=PRECISION_DIR,
    output_root=DRIVE_TASK_DIR,
    registry_path=DRIVE_REGISTRY,
    registry_mirrors=(LOCAL_REGISTRY,),
    root=REPO_DIR,
)
print("Screen:", result["status"])
print("Label basis:", result.get("comparison_label_basis"))
for row in result.get("folds", []):
    print(
        "Fold",
        row["fold"],
        "train F1:",
        row["candidate_train_f1"],
        "validation F1:",
        row["candidate_validation_f1"],
        "gap:",
        row["candidate_gap"],
    )
for gate in result.get("checks", []):
    if gate["status"] != "pass":
        print(gate)
if "reason" in result:
    print(result["reason"])
if "incremental_comparison" in result:
    incremental = result["incremental_comparison"]
    print("Direct name-truth parents:", result["direct_parent_run_ids"])
    print("F1 change versus NameTruth on the SAME new labels:", incremental["validation_delta"])
    print("Paired 95% interval:", incremental["validation_interval"])
    print("Class F1 changes:", incremental["class_f1_delta"])
    print("Induced corruption changes:", incremental["mean_induced_change_delta"])
if "incremental_comparison" in result:
    for row in result["incremental_comparison"]["folds"]:
        print("Direct-parent gap change, fold", row["fold"], ":", row["delta_gap"])

## 5. Stop and review

Stop after folds 0 and 4. Do not auto-run folds 1–3, refit or open the held-out test. Every fit is recorded in `results/runs.csv` through the shared trainer. Reuse requires matching labels, source runs, configuration, MixUp training evidence and artifact hashes. A memory failure stops before another fold starts.

Results are saved under `MyDrive/MLA2/task3/experiments/t3_gender_name_truth_mixup_alpha020/gender`:

- Each run's `mixup_training.json`: frozen policy and training-row hash, plus per-epoch row counts, batch counts, lambda sum, self/same-label pair counts and mixing-plan hash. The trainer checks every original row was used once each epoch.
- Each run's `history.csv`: mixed training loss and ordinary validation scores. Online training F1 is blank, with an explicit mixed-input scope.
- `screen_decision.json`: the unchanged 19 checks versus matched G2/E6, plus direct NameTruth parent IDs.
- `incremental_comparison.json`: the effect of MixUp versus completed 04ad. Existing `dropout_*` fields refer to these NameTruth parents. Read per-class scores, clean gaps and raw corruption scores together.
- `clean_gap_comparison.csv` and `ieee_oof_predictions.csv`: clean scores and validation probabilities.
- `label_basis_comparison.csv` and `original_label_diagnostic.json`: score identical probabilities against name-truth and teacher labels, with no extra GPU pass or new gate.
- `source_audit.json`, `label_variant/` and `comparison_name_truth_ieee/`: parent and MixUp contracts, label evidence, and separate matched evaluations.

Look for a smaller **clean** gap while retaining validation and rare-class scores. A lower mixed training score alone is not evidence of less overfitting. This is one trial, not a search over MixUp strengths. A pass does not accept a final model.
